# Learning of filter (version 2)

### before we begin, 
is all this at all useful or necessary? What are the existing methods of projection-domain denoising?

### Overall plan
Want to create a neural network which takes in a noised sinogram $\mathcal{R}f = S(\alpha, r)$ and outputs a filter window $W(\omega)$ which optimally suppresses the specific noise in $S$ (while of course also minimizing reconstruction error) during FBP $f = \mathcal{R}^*(\mathcal{F}^{-1} (|\omega|W(\omega)\mathcal{F}(\mathcal{R}f)))$.

### Training dataset 
- start by generating ~100 ellipse phantoms
- apply FORMAL forward radon for ellipses, and sample to create sinogram (so that there's no discretization error).
- add various mixtures of the two types of noise common in CT applications (Gaussian and Poisson) to get ~1500 noised sinograms.
- train on 1200, reserve 300 for evaluation

### implemetation of FBP
- using our own implementation, which works the same way as skimage iRadon (see transform.py)

### training procedure
- initialize $W(\omega)$ (as the identity, as various common filters, to observe how the filter converges from different starting points)

### Loss function
- compare original ground truth ellipse phantome to reconstruction of noised sinogram.
- start with MSE
- Probably want a more sophisticated measure that penalizes edge-blur and is not biased towards over-smoothing.
- 

### network architecture
- CNN or some other transformer architecture. Gradient descent on... what? what do we paramaterize in order to perturb the function $W$ so that it loss function is improved?


### Crazy idea,
- Convolutional Neural networks can learn features, such as edges and certain textures, which occur inside the KxK kernel window size, right? We slide this window along to scope the entire image, and each node in the next layer corresponds to a different kernel which we are training. 

I want to apply a CNN to siongrams in order to detect noise vs actual data, and filter out the noise. 

But when we're looking at a sinogram, we're not looking directly at the features (edges, voids) in the image. An object in the image (which is off-center from the detextor axis) corresponds to a sine wave in the sinogram. (eg, for any one pixel in the image, there is a corresponding sine wave in the sinogram which is all the projections incident to the pixel.) 

What if we made a CNN that used sinusoidal-shaped kernel windows? 

(or is it a better idea to still use square kernels, but have them learn to be sensitive to sinusoidal shaped patterns within?)

And, (at this point I know my intuition is not correct), if we have a fixed detector grid fineness, we can set an upper limit on the frequencies of sine wave we'd like to check for? 

Also the reason we don't just do a CNN which acts on the fourier-transformed siongram (taking 1d fourier transforms along the detector axis) is because the noise is going the flood the fourier transform with high frequencies, so we want to do this CNN that acts on the sinogram BEFORE fourier transformation, so that we can apply a filter first.

# Learning of filter (version 1)

### Training data
- lots of noised sinograms and original images

### loss function
- MSE between reconstructed image and original image, as starting point.
- Then can introduce a more intricate metric, or a combination of metrics, to avoid MSE-specific biases.

### learning paramaters
- We are training a noise-supression/windowing function that is multiplied by the standard ramp filter during back projection.
- ~~start with a latent grid of ~400 (mirrored) gaussians, of varying mean and variance,~~
- ~~paramaters are the 400 weights of each Gaussian.~~
- Allow negative weights. 
- ~~(Consider regularization or filtering of some sort, since the gaussians will be highly overlapping. So that the learning doesn't have too many degrees of freedom to where it gets lost and is unable to choose a direction to optimize toward.)~~

### Learning architecture
- gradient decsent and **(one-step)** backprop 
- compute the gradient of the loss function with respect to each paramater,
- ~~backrpopogate according to ADAM or some other gradient descent procedure.~~
- Since everything is linear, "one-step backprpop using a MSE loss function" is just least squares regression. 
- ~~(could nievergelt's elemtary inversion technique be helpful in allowing us to take gradients of the filtered back projection?)~~ 

## ~~Filter Learning Ideas~~

Ok, to develop a little more my inclinations here, we want to learn a filter which is a gaussin mixture, by using gradient descent backprop. 

Training set would be paired oringal images and noised sinograms. ~~Can use online datasets which already include paired sinograms and reconstructions (so assuming they're accurate and treating the CT image as if it's the original image.) Can also generate our own bitmap images, or use any random bitmap image.~~ Generate our own ellipse phantoms. Then, can use a number of noise generators with different paramaters. So our data set is very easily expandable. (can use the same image with multiple different layered noise treatments.)

*Instead of using numerical forward Radon transform, can symbolically compute Radon transform of known function of ellipses, and then sample from that function to get a sinogram without discretization artifacts.*

*"(SOURCE: ) For an ellipse with semi-axes $a,b$, rotation $\alpha$, center $(x_0,y_0)$, and density $\rho_0$,*

\begin{align*} g(\theta,s) = \frac{2\rho_0 ab}{a_\theta^2}\sqrt{a_\theta^2 - (s-s_0)^2} \end{align*} 

*for $|s-s_0|<a_\theta$ and 0 otherwise. Here $a_\theta^2 = a^2\cos^2(\theta-\alpha)+b^2\sin^2(\theta-\alpha)$ and $s_0 = x_0\cos\theta + y_0\sin\theta$."*

Can implement our own ~~forward radon transform and~~ filtered back projection, so that we can choose the filter, and also so that we can ensure the transforms are differentiable. 

Noising can be informed by actual CT sensor noise. Can do training runs on differnt data sets of mixed or only one type of noise, and differing magnitudes of noise. Then can evaluate learned filters performance on different kinds of noise.

We want to develop a filter that is good at surpressing noise, as well as accurately doing FBP without artifacts. 

Bell- shaped filters are good at the first, ramp-shaped filters are good at the second. What is good at both? 

Include the standard ramp filter, and then multiply by a noise-supressing filter window on top of that: $H(\omega) = |\omega| W(\omega)$, where $W(\omega)$ is the mixture.

*(Also, (SOURCE: ) Ramp filter is 0 at the origin, which would give our sinogram a DC bias (value at (0,0) in the fourier space) of 0. To avoid this bias and assign the correct non-zero DC value, instead of naively sampling $\vert{}\omega\vert{}$ as a point-wise function, we want to derive the discrete filter from the exact periodic/discrete duality or using the FFT of the proper band-limited spatial convolution kernel.)*

Assume the goal filter window is ~~a gaussin mixture.~~ (Or rather, a "difference of gaussians," so that negative weights are allowed.) ~~Furthermore, can limit number of gaussians, to say 6. Where 3 of them have man 0, and the other 3 are mirrored accross 0. Paramters are 6 variances, 3 non-zero means, and 6 weights. (Not sure how we could possibly implement a capacity to add more filters as a trainable parameter.)~~

Train $W(\omega)$ by initializing a large finite 2d grid of gaussians of varying means and variances. Then the paramaters can be the weight of each gaussian. (and allow negative weights.)

Perform ~~backprop (1 step) and gradient descent,~~ multilinear least squares, utilizing different subsets of the training data. 

Can use various standard filters (or rather, gaussian mixture approximations therof) as starting points and see if the learned filters converge.

The likelihood that we find some undiscovered effective filter shape is probably pretty low; rather, the main goal of this exercise would be to demonstrate that our learning procedure converges to one/multiple of the commonly used filters.




## ~~Next step: filter neural network~~

(COMPARE WITH "WEINER WINDOW.")

So in the above, our approach is to initialize a filter 

\begin{align*} H(\omega) = |\omega| W(\theta; \omega) \end{align*}

where $W(\omega)$ is the learnable gaussian mixture. This filter applies the same way to any input sinogram.

Next step: replace the gaussian mixture function with some neural network, say a MLP:

\begin{align*} H(\omega) = |\omega| R(\theta; \omega, S) \end{align*}

which can also learn to modulate itself based on the properties of the sinogram as a whole. 

~~The input layer will then not just be the individual input sinograms,~~

Or rather, could we use a CNN? So that the input projections are not just considered individually, but also in groups, so that we can get something that is responsive to large-scale features in the sinogram? 

- (Another question, could we make the filter depend on the projection angle as well, so that it can learn to identify and deal with angle-dependent noise? Does this require an expanded input layer, or can this also be achieved by using a CNN?) 

- "it can be observed that a certain reconstruction point only relates to a few radon projected points, which constitute a sinusoid in radon projection domain. This allows us to construct an appropriative learnable operation for back-projection with far fewer parameters than the fully-connected operation. This is denoted as a sinusoidal back-projection layer." (SOURCE: HeMa) So this indicates we should look at using sinusoidal-shaped windowing for the CNN.

## ~~In-between approach:~~


## ~~Application with regard to reconstruction of probability distributions from marginal densities~~

(RESEARCH "ADAPTIVE WEINER FILTERING.")

We train a neural network, but the neural network is not the filter. Rather, the neural network controls a paramaterization, selecting from a family of pre-set filters. This would likely involve first creating a functional of filters which optimizes MSE for any known mixture of noises. And then the main purpose of the neural network would be to predict the mixture of noises based on sinogram input data.

(In any case, can separate whatever algorithm/network we use into two stages: First assessment of noise, and then selection of optimal filter for estimated noise. If doing a neural network, maybe we can insert a script into the beginning layers that also outputs "this is what my assessment of the noise is.")

Maybe we don't even need to do deep learning, and can instead hard code the estimator function which takes in a sinogram and estimates the mix of a known library of noises. (can still write it in terms of a network of nodes, and leverage convolution) and then derive the function by hand which selects the optimal filter given a set mix of noise.

- (Can we derive some sort of approximate to cramer-wold which guaruntees our construction is within a certain bound, dependent on the number (and arrangement) of projections?)

Next we want to turn our sights away from medical imaging CT and toward reconstruction of probability distributions from marginal data. 

Can we use the same procedure (data syntesis and then macine learning) to train a filter window which suppresses statistical noise? 

Is the neural network ramp filter we trained on medical imaging data effective at filtering statistical noise out-of-the-box? Conversely, if we train a neural network only on probability distribution data, can it effectively filter CT noise? 